# Laboratorio Snowpark DataFrames - Mediaset

In questo laboratorio esploreremo le funzionalità di **Snowpark Python DataFrames** utilizzando dati reali del palinsesto televisivo Mediaset.

Impareremo ad utilizzare le operazioni fondamentali:
- **`select`** - Selezione di colonne specifiche
- **`filter`** - Filtraggio di righe con condizioni
- **`group_by`** e **`agg`** - Aggregazioni e raggruppamenti
- **`join`** - Unione di più DataFrame
- **Window Functions** - Funzioni finestra per ranking e analisi avanzate
- **`save_as_table`** - Salvataggio dei risultati in tabelle Snowflake

I dati sono contenuti nel database `MEDIASET_LAB`, schema `RAW`, e comprendono informazioni su programmi TV, ascolti e palinsesto.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.use_database("MEDIASET_LAB")
session.use_schema("RAW")
print("Sessione Snowpark attiva!")

## 1. Caricamento DataFrame

Il primo passo è caricare i dati dalle tabelle Snowflake in DataFrame Snowpark. A differenza di Pandas, i DataFrame Snowpark sono **lazy**: le operazioni vengono eseguite solo quando richiesto (ad esempio con `.show()` o `.collect()`).

Utilizziamo `session.table()` per creare un riferimento a ciascuna tabella.

In [ ]:
df_programmi = session.table("MEDIASET_LAB.RAW.PROGRAMMI_TV")
df_ascolti = session.table("MEDIASET_LAB.RAW.ASCOLTI")
df_palinsesto = session.table("MEDIASET_LAB.RAW.PALINSESTO")

print(f"Programmi TV: {df_programmi.count()} righe")
print(f"Ascolti: {df_ascolti.count()} righe")
print(f"Palinsesto: {df_palinsesto.count()} righe")

df_programmi.show()

## 2. Operazioni Base: Select e Filter

Le operazioni `select` e `filter` sono le più comuni quando si lavora con i DataFrame:
- **`select()`** permette di scegliere solo le colonne di interesse
- **`filter()`** permette di applicare condizioni per selezionare sottoinsiemi di righe

Utilizziamo `col()` per riferirci alle colonne e costruire espressioni di filtro.

In [ ]:
from snowflake.snowpark.functions import col

# Select colonne specifiche
df_programmi.select("TITOLO", "GENERE", "CANALE", "COSTO_EPISODIO_EUR").show()

# Filter: programmi su Canale 5
df_canale5 = df_programmi.filter(col("CANALE") == "Canale 5")
print(f"Programmi su Canale 5: {df_canale5.count()}")
df_canale5.select("TITOLO", "GENERE", "COSTO_EPISODIO_EUR").show()

# Filter multiplo: programmi Reality con costo > 100000
df_reality_costosi = df_programmi.filter(
    (col("GENERE") == "Reality") & (col("COSTO_EPISODIO_EUR") > 100000)
)
df_reality_costosi.select("TITOLO", "COSTO_EPISODIO_EUR").show()

## 3. Aggregazioni: Group By e Agg

Le aggregazioni permettono di calcolare metriche riassuntive raggruppando i dati per una o più colonne.

Utilizziamo:
- **`group_by()`** per definire le colonne di raggruppamento
- **`agg()`** per specificare le funzioni di aggregazione (`count`, `avg`, `sum`, ecc.)
- **`alias()`** per rinominare le colonne risultanti

In [ ]:
from snowflake.snowpark.functions import avg, sum as sum_, count, round as round_

# Numero programmi per genere
df_programmi.group_by("GENERE").agg(
    count("*").alias("NUM_PROGRAMMI"),
    round_(avg("COSTO_EPISODIO_EUR"), 0).alias("COSTO_MEDIO")
).sort(col("NUM_PROGRAMMI").desc()).show()

# Share medio per canale
df_ascolti.join(df_programmi, "PROGRAMMA_ID").group_by("CANALE").agg(
    round_(avg("SHARE_PERCENTUALE"), 2).alias("SHARE_MEDIO"),
    sum_("TELESPETTATORI").alias("TELESPETTATORI_TOTALI"),
    count("*").alias("NUM_RILEVAZIONI")
).sort(col("SHARE_MEDIO").desc()).show()

## 4. Join tra DataFrame

Il `join` permette di combinare dati provenienti da tabelle diverse. In Snowpark, possiamo eseguire join specificando la colonna di collegamento.

Uniamo i dati di **programmi** e **ascolti** per identificare i programmi con il miglior share medio, e analizziamo gli ascolti per fascia oraria e regione.

In [ ]:
# Join programmi con ascolti - top 10 programmi per share
df_top = df_ascolti.join(df_programmi, "PROGRAMMA_ID") \
    .group_by("TITOLO", "GENERE", "CANALE") \
    .agg(
        round_(avg("SHARE_PERCENTUALE"), 2).alias("SHARE_MEDIO"),
        sum_("TELESPETTATORI").alias("TELESPETTATORI_TOTALI"),
        count("*").alias("RILEVAZIONI")
    ) \
    .sort(col("SHARE_MEDIO").desc()) \
    .limit(10)

df_top.show()

In [ ]:
# Ascolti per fascia oraria
df_ascolti.group_by("FASCIA_ORARIA").agg(
    round_(avg("SHARE_PERCENTUALE"), 2).alias("SHARE_MEDIO"),
    sum_("TELESPETTATORI").alias("TELESPETTATORI_TOTALI")
).sort(col("SHARE_MEDIO").desc()).show()

# Ascolti per regione
df_ascolti.group_by("REGIONE").agg(
    round_(avg("SHARE_PERCENTUALE"), 2).alias("SHARE_MEDIO"),
    sum_("TELESPETTATORI").alias("TELESPETTATORI_TOTALI")
).sort(col("TELESPETTATORI_TOTALI").desc()).show()

## 5. Window Functions

Le **Window Functions** (funzioni finestra) permettono di eseguire calcoli su un insieme di righe correlate alla riga corrente, senza ridurre il numero di righe nel risultato.

Utilizziamo `Window.partition_by()` per definire le partizioni e `rank()` per creare una classifica dei programmi all'interno di ciascun canale.

In [ ]:
from snowflake.snowpark.functions import rank
from snowflake.snowpark import Window

# Ranking programmi per share medio all'interno di ogni canale
window_spec = Window.partition_by("CANALE").order_by(col("SHARE_MEDIO").desc())

df_ranking = df_ascolti.join(df_programmi, "PROGRAMMA_ID") \
    .group_by("TITOLO", "CANALE") \
    .agg(round_(avg("SHARE_PERCENTUALE"), 2).alias("SHARE_MEDIO")) \
    .with_column("RANK_IN_CANALE", rank().over(window_spec))

df_ranking.filter(col("RANK_IN_CANALE") <= 3).sort("CANALE", "RANK_IN_CANALE").show()

## 6. Salvataggio Risultati

Una volta completata l'analisi, possiamo salvare i risultati direttamente in una tabella Snowflake utilizzando `save_as_table()`. Questo è particolarmente utile per creare tabelle di reporting o per alimentare dashboard.

Salviamo la classifica dei top programmi nello schema `ANALYTICS`.

In [ ]:
# Salva la classifica Top Programmi nello schema ANALYTICS
df_top.write.mode("overwrite").save_as_table("MEDIASET_LAB.ANALYTICS.TOP_PROGRAMMI_SHARE")

# Verifica
session.table("MEDIASET_LAB.ANALYTICS.TOP_PROGRAMMI_SHARE").show()
print("Tabella MEDIASET_LAB.ANALYTICS.TOP_PROGRAMMI_SHARE creata con successo!")

## 7. Visualizzazione con Matplotlib

Snowpark si integra con le librerie di visualizzazione Python. Convertiamo i DataFrame Snowpark in Pandas con `.to_pandas()` e utilizziamo **Matplotlib** per creare grafici.

In [ ]:
import matplotlib.pyplot as plt

# Converti in Pandas per la visualizzazione
pdf_top = df_top.to_pandas()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(pdf_top["TITOLO"], pdf_top["SHARE_MEDIO"], color="#29B5E8")
ax.set_xlabel("Share Medio (%)")
ax.set_title("Top 10 Programmi Mediaset per Share Medio")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
pdf_generi = df_programmi.group_by("GENERE").agg(
    count("*").alias("NUM_PROGRAMMI")
).to_pandas()

fig, ax = plt.subplots(figsize=(8, 8))
colors = ["#29B5E8", "#11567F", "#1B8BBE", "#64C8F0", "#0D3F5C", "#87D4F2", "#073A56", "#A5E0F5", "#1A7BA8", "#C0ECF9"]
ax.pie(pdf_generi["NUM_PROGRAMMI"], labels=pdf_generi["GENERE"], autopct="%1.0f%%",
       colors=colors[:len(pdf_generi)], startangle=90)
ax.set_title("Distribuzione Programmi per Genere")
plt.tight_layout()
plt.show()

## Conclusioni

In questo laboratorio abbiamo esplorato le principali funzionalità di **Snowpark Python DataFrames**:

| Operazione | Descrizione |
|---|---|
| `session.table()` | Caricamento dati da tabelle Snowflake |
| `select()` | Selezione di colonne specifiche |
| `filter()` | Filtraggio righe con condizioni |
| `group_by()` + `agg()` | Aggregazioni e raggruppamenti |
| `join()` | Unione di DataFrame su colonne comuni |
| `Window` + `rank()` | Funzioni finestra per classifiche e analisi avanzate |
| `save_as_table()` | Salvataggio risultati in tabelle Snowflake |
| `.to_pandas()` | Conversione in Pandas per visualizzazioni |

Tutte le operazioni vengono eseguite direttamente nel motore Snowflake, sfruttando la potenza di calcolo del cloud senza necessità di spostare i dati.